# Week 8 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-08.md`](../../weeks/week-08.md) · **Given engine:** [`unify.py`](unify.py)

This notebook is the **teaching walkthrough** of the week-8 studio (unification, the
occurs check, forward chaining). It **imports the reference code from
[`solution.py`](solution.py)** — it never re-pastes it — so the answer shown here is
byte-for-byte the one the provided test grades. The correctness guarantee is separate:
`python3 studios/_verify_solutions.py week-08` runs the unmodified `test_unify.py`
against `solution.py`.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [ ]:
# --- bootstrap: put the week folder (for solution/unify) and the repo root
# (for aicourse) on the path, so this runs from anywhere. ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-08"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import unify
from unify import Var, Compound, unify as u, unify_no_occurs, resolved_mgu, is_compound, LECTURE_CASES, FACTS, RULES, X, f
import solution

## Task 1 — the occurs check (rule 4)

`occurs(var, x, theta)` answers *does `var` appear inside `x`, chasing bindings in
`theta`?* Three cases: the variable itself, a bound variable (recurse), and a compound
(any argument). `unify_var` calls it before binding, so returning `True` here is what
**refuses** a cyclic binding. Straight from `solution.py`:

In [ ]:
print(inspect.getsource(solution.occurs))

# Mirror test_student_occurs_matches_expected: all three cases, incl. through theta.
A, B, C = Var("A"), Var("B"), Var("C")
def _c(op, *args): return Compound(op, *args)
cases = [
    (A, A, {},                          True),
    (A, _c("f", A), {},                 True),
    (A, _c("f", B), {},                 False),
    (A, "tom", {},                      False),
    (A, B, {},                          False),
    (A, B, {B: _c("f", A)},             True),   # occurs THROUGH a binding
    (A, _c("g", _c("h", B), C), {C: A}, True),   # occurs via a bound arg
]
for var, x, theta, expected in cases:
    got = solution.occurs(var, x, theta)
    assert got == expected, (var, x, theta, got, expected)
    print(f"occurs({var}, {x}, {theta}) = {got}")
print("\nall occurs cases match")

## Task 1, the guarantee fails on request — `unify(X, f(X))`

This is the payload of week 8. **With** the occurs check, `unify(X, f(X))` returns
`None` — the binding `X = f(X)` is refused. **Without** it (`unify_no_occurs`, i.e.
what standard Prolog does for speed), the binding is accepted and unrolls into the
infinite term `f(f(f(...)))`. Two lines of code, and a formal soundness guarantee is
gone. We watch it break, on purpose.

In [ ]:
# WITH the occurs check: refused. This is the whole guarantee.
assert u(X, f(X)) is None
print("unify(X, f(X)) WITH occurs check   =", u(X, f(X)), " <-- refused (sound)")

# WITHOUT it: the self-referential binding is accepted.
theta = unify_no_occurs(X, f(X))
bound = theta[X]
assert is_compound(bound) and bound.op == "f" and bound.args[0] == X
print("unify_no_occurs(X, f(X)) binds X to", bound, " <-- accepted (Prolog does this)")

# Follow the self-reference a few steps by hand: it never terminates.
term = bound
for _ in range(4):
    term = Compound("f", term)
depth, node = 0, term
while is_compound(node):
    node, depth = node.args[0], depth + 1
print(f"substituted term keeps growing without bound (walked {depth} levels of f(...))")
print("\nGUARANTEE FAIL confirmed: drop the occurs check and unification becomes unsound.")

## Task 2 — forward chaining over the family KB

`forward_chain(facts, rules)` computes the deductive closure to a **fixpoint**: each
pass matches every rule body against a frozen snapshot of the facts (via the GIVEN
`unify`) and adds the now-ground head, repeating until nothing new appears. The
recursive `ancestor` rule is exactly why several passes are needed. Same knowledge
Prolog answers *backward*, computed *forward*.

In [ ]:
print(inspect.getsource(solution.forward_chain))
print(inspect.getsource(solution._match))

In [ ]:
# Mirror test_forward_chain_derives_family_closure.
closure = set(solution.forward_chain(FACTS, RULES))

must_contain = [
    _c("father", "tom", "bob"), _c("father", "bob", "ann"), _c("father", "bob", "pat"),
    _c("grandparent", "tom", "ann"), _c("grandparent", "tom", "pat"),
    _c("ancestor", "tom", "bob"), _c("ancestor", "bob", "ann"),
    _c("ancestor", "tom", "ann"), _c("ancestor", "tom", "pat"),   # recursive steps
    _c("inherits", "bird", "flies"), _c("inherits", "bluebird", "flies"),  # via isa chain
]
for fact in must_contain:
    assert fact in closure, fact
must_not_contain = [
    _c("father", "tom", "ann"), _c("ancestor", "ann", "tom"),
    _c("inherits", "vertebrate", "flies"),
]
for fact in must_not_contain:
    assert fact not in closure, fact
assert set(FACTS) <= closure   # monotone: never drops a known fact

print(f"closure derived {len(closure) - len(FACTS)} new facts:")
for a in sorted(str(x) for x in closure - set(FACTS)):
    print("  ", a)

### Forward vs. backward — the discussion deliverable

Forward chaining derived everything above in a few passes; it is the right direction
for *"what is all-derivable?"*. Backward chaining (Prolog) is goal-driven — right for
*"can THIS hold?"* — but risks non-termination on left recursion (`unify.BROKEN_KB`
reorders the `ancestor` rule: identical logic, broken control). Each direction owns a
different failure mode. `Algorithm = Logic + Control`.

## Task 3 — NL → FOL → prover is Duel 2 (headless demo)

Task 3 is the launch of **Duel 2 — The Formalizer**: the LLM translates natural
language to Prolog, and a **sound prover** (`swipl`) decides the answer — never the
model's own verdict. Here we run the model call headless with the deterministic
`echo` backend so the notebook executes anywhere.

> **The `echo` backend is a FAKE** — its text is not a model result. For a real Duel 2
> run, re-run with `backend="ollama"` (or `"manual"`) and feed the output to `swipl`
> via `unify.run_prolog`. **Never trust the model's verdict; run the prover.**

In [ ]:
from aicourse.llm import LLM
llm = LLM(backend="echo")   # echo = headless-safe FAKE

premises = "All men are mortal. Socrates is a man."
question = "Is Socrates mortal?"
arm_b_prompt = (
    "Translate to Prolog. Output only clauses, one per line, ending with a period.\n"
    "Use lowercase for constants and predicates, uppercase for variables.\n"
    "Then output the query on the last line prefixed with '?- '.\n\n"
    f"Text: {premises}\nQuestion: {question}\n"
)
resp = llm.complete(arm_b_prompt)
print("Arm B (LLM as translator) — model output:")
print(" ", resp.text)
print("\n[reminder] echo is a FAKE. The verdict comes from swipl on the translated")
print("clauses, not from the model. 'No answer' (syntax error / non-termination) is a")
print("legitimate third outcome this week — FOL is semi-decidable (scorecard axis 8).")